In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.colors import LogNorm

from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy
import warnings

from matplotlib.colors import LogNorm, Normalize
# from scipy.interpolate import interp1d  
from scipy import stats
from matplotlib.ticker import LogLocator
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="Tracked_Ascent_Trajectories",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
#Get Data
# def GetData_V1(trackedArray):
#     pIndices=trackedArray[:,0]
    
#     varNames=GetVarNames()
    
#     dataDictionary = {varName: np.full((len(ModelData.timeStrings), len(pIndices)), np.nan) for varName in varNames}
    
#     for t in tqdm(range(ModelData.Ntime), position=2, leave=False):
#         for varName in varNames:
#             if any(prefix in var for var in varNames for prefix in ("E_")) and t == ModelData.Ntime-1:
#                 dataDictionary[varName][t] = np.full((1, len(pIndices)), np.nan)
#             else:
#                 ###############
#                 dataDictionary[varName][t] = CallLagrangianArray(ModelData, DataManager, \
#                                                                  ModelData.timeStrings[t], varName)[pIndices]
#                 ###############

#     return dataDictionary

def GetData_V2(trackedArray):
    pIndices=trackedArray[:,0]
    
    varNames=GetVarNames()
    dataDictionary = {varName: np.full((len(ModelData.timeStrings), len(pIndices)), np.nan) for varName in varNames}
    
    for t in tqdm(range(ModelData.Ntime), position=2, leave=False):
        dataLocationInfoCache = {'inputDataDirectory': None, 'dataName': None}; currentFile = None
        
        for varName in varNames:
            if ("E_" in varName or "D_" in varName) and t == ModelData.Ntime-1:
                dataDictionary[varName][t] = np.full((1, len(pIndices)), np.nan)
            else:
                ###############
                [inputDataDirectory,dataName] = CallLagrangianArray(ModelData, DataManager, \
                                                                    ModelData.timeStrings[t], \
                                                                    varName,loadData=False) #get file info
                if inputDataDirectory != dataLocationInfoCache['inputDataDirectory']\
                or dataName != dataLocationInfoCache['dataName']: #if not in cache, open new file
                    # print('reloading',varName)
                    if currentFile is not None: currentFile.close() #close the old file
                    currentFile=DataManager.GetTimestepData_V2(inputDataDirectory, ModelData.timeStrings[t], dataName)
                    dataLocationInfoCache['inputDataDirectory']=inputDataDirectory #set cache
                    dataLocationInfoCache['dataName']=dataName #set cache
                dataDictionary[varName][t] = currentFile[varName][:][pIndices] #grab the needed variable
                ###############
        if currentFile is not None: currentFile.close() #close the file at end of each timestep
    return dataDictionary

#Entrainment Variable Things
def GetVarNames():
    varNames = ['z','Z','LFC','QCQI','W']#,'A_g','A_c']
    if dataName == 'Variables':
        varNames += ['VMF_g','VMF_c','BUOYANCY2']
        varNames += ['QV','QCQI','RH_vapor','HMC']
        varNames += ['THETA_v','THETA_e']
        varNames += ['X']
        
    elif "Entrainment" in dataName:
        PROCESSED_string = "PROCESSED_"
        varNames += ['Z']
        for updraftType in ['g','c']:
            for DivideMassFlux_string in ["","_DivideMassFlux","_DivideMassFluxMean"]:
                varNames += [f'{PROCESSED_string}E{DivideMassFlux_string}_{updraftType}',f'{PROCESSED_string}D{DivideMassFlux_string}_{updraftType}']
                
    elif "UpdraftArea" in dataName:
        for updraftType in ['g','c']:
            varNames += [f'UPDRAFT_AREA_{updraftType}',f'UPDRAFT_EDGE_DISTANCE_{updraftType}']
    return varNames

def GetMeanVarNames():
    varNames=[]
    if dataName == 'Variables':
        varNames+=['QV','RH_vapor','HMC']
        varNames+=['THETA_v','THETA_e']
    return varNames

In [ ]:
##############################################
#COMPUTING FUNCTIONS

In [ ]:
def RunCode(trackedArrays,parcelTypes):

    parcelDepths=['ALL']
    
    #subset out parcelType and parcelDepth
    trajectoriesArrayDictionary = {pt: {pd: {} for pd in parcelDepths} for pt in parcelTypes}
    priorToAscentArrayDictionary = {pt: {pd: {} for pd in parcelDepths} for pt in parcelTypes}
    for parcelType in tqdm(parcelTypes, position=0, leave=True):
        for parcelDepth in tqdm(parcelDepths, desc=f"Type: {parcelType}", position=1, leave=False):

            #subsetting data
            trackedArray = trackedArrays[parcelType][parcelDepth]
            t1 = trackedArray[:, 1].astype(int); t2 = trackedArray[:, 2].astype(int)
            after = trackedArray[:, 3].astype(int)
        
            # Data Calculations
            # loading variable data dictionary
            tqdm.write(f"Getting Data")
            dataDictionary = GetData_V2(trackedArray)
            
            #getting data
            varNames=GetVarNames()
            for varName in tqdm(varNames, desc="Subsetting Variable", leave=False):
                variableData = dataDictionary[varName]
                
                #computing trajectories for each variable
                trajectoriesArray = SubsetTrajectories(variableData,t1,t2,after)
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName] = trajectoriesArray
                
                priorToAscentArray = SubsetTrajectories(variableData,t1=0,t2=t1,after=0)
                priorToAscentArrayDictionary[parcelType][parcelDepth][varName] = priorToAscentArray

    return trajectoriesArrayDictionary,priorToAscentArrayDictionary

def SubsetTrajectories(variableData,t1,t2,after):
    #making output array
    trajectoriesArray = np.full_like(variableData,fill_value=np.nan,dtype=float)

    #masking the final output array
    time_idx = np.arange(variableData.shape[0])[:, np.newaxis]
    mask = (time_idx >= t1) & (time_idx <= t2+after)
    
    #applying mask to output array
    trajectoriesArray[mask] = variableData[mask]

    return trajectoriesArray

def LimitTrackedArraysRows(trackedArrays, limit=None): #limit=(0,10000)
    if limit is None:
        return trackedArrays
    for parcelType in trackedArrays:
        for parcelDepth in trackedArrays[parcelType]:
            trackedArrays[parcelType][parcelDepth] \
            = trackedArrays[parcelType][parcelDepth][limit[0]:limit[1], :]
    return trackedArrays
    
# #estimating file size
# Np=107076; Nt=661; Nvars=5;
# Nparceltypes=2
# Np*Nt*Nvars*Nparceltypes*4/1e9


In [ ]:
##############################################
#COMPUTING
running=True #KEEP TRUE WHEN JOBARRAY IS RUNNING
running=False

In [ ]:
if running:
    #Loading in Tracked Parcels Info
    trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class)
    # trackedArrays=LimitTrackedArraysRows(trackedArrays) #limiting number of parcels to allow computation to complete

    parcelTypesList = [['CL','nonCL']]
    for parcelTypes in parcelTypesList:
        [trajectoriesArrayDictionary,priorToAscentArrayDictionary] = RunCode(trackedArrays,parcelTypes)
        TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles,trajectoriesArrayDictionary, dataName, t=f'trajectoriesArray_{parcelTypes[0]}vs{parcelTypes[1]}')
        TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles,priorToAscentArrayDictionary, dataName, t=f'priorToAscent_{parcelTypes[0]}vs{parcelTypes[1]}')

In [ ]:
##############################################
#DATA LOADING AND POST-PROCESSING FUNCTIONS

In [ ]:
#Loading SBF X Locations

def Get_AvgConvergence(t):

    timeString = ModelData.timeStrings[t]
    outputDataDirectory=os.path.normpath(os.path.join(DataManager.outputDataDirectory,"..","Eulerian_CLTracking"))
    Dictionary = TrackingAlgorithms_DataLoading_Class.LoadData(ModelData, DataManager, timeString,
                     dataName="Eulerian_CLTracking",outputDataDirectory=outputDataDirectory,printstatement=False)
    avgConvergence = Dictionary["avgConvergence"]
    return avgConvergence
    
def find_SBF_xmaxs():
    xmaxs=[]
    for t in tqdm(range(ModelData.Ntime)):
        if t == 0:
            xmaxs.append(np.nan)
        else:
            avgConvergence = Get_AvgConvergence(t)
            avgConvergence_max=np.nanmax(avgConvergence)
            xmax = np.where(avgConvergence==avgConvergence_max)[0][0]
            xmaxs.append(xmax)
    return xmaxs

def Get_SBF_xmaxs_FilePath(ModelData, DataManager):
    fileName = (
        f"SBF_xmaxs_{ModelData.res}_{ModelData.t_res}_"
        f"{ModelData.Ntime}nt.pkl"
    )
    return os.path.join(DataManager.outputDataDirectory, fileName)

def LoadOrRun_find_SBF_xmaxs(ModelData, DataManager, overwrite=False):
    filePath = Get_SBF_xmaxs_FilePath(ModelData, DataManager)
    
    if os.path.exists(filePath) and not overwrite:
        print(f"reading from {filePath}")
        with open(filePath, "rb") as f:
            xmaxs = pickle.load(f)
        return xmaxs

    xmaxs = find_SBF_xmaxs()

    os.makedirs(os.path.dirname(filePath), exist_ok=True)
    with open(filePath, "wb") as f:
        pickle.dump(xmaxs, f)
    print(f"saved to {filePath}")
    return xmaxs

SBF_XLocations = LoadOrRun_find_SBF_xmaxs(ModelData, DataManager)

In [ ]:
#Data Loading and Post-Processing Functions
def LoadAndProcessData(subsetting_RightOfSBF=False,subsetting_time=None,dataName=dataName):

    parcelTypesList = [['CL','nonCL']]; parcelTypes = parcelTypesList[0]
    trajectoriesArrayDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=f'trajectoriesArray_{parcelTypes[0]}vs{parcelTypes[1]}')
    priorToAscentArrayDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=f'priorToAscent_{parcelTypes[0]}vs{parcelTypes[1]}')
    SubtractMeanData(trajectoriesArrayDictionary,priorToAscentArrayDictionary,dataName)
    if dataName == 'Variables':
        SubtractSBF_XLocation(trajectoriesArrayDictionary,priorToAscentArrayDictionary,SBF_XLocations)

    if dataName == 'Entrainment':
        print(dataName)
        FixDetrainmentSign(trajectoriesArrayDictionary); FixDetrainmentSign(priorToAscentArrayDictionary)
        CalculateNetEntrainment(trajectoriesArrayDictionary); CalculateNetEntrainment(priorToAscentArrayDictionary)

    #subsetting only right of SBF
    if subsetting_RightOfSBF:
        trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class)
        LimitLeftOrRightOfSBF(trajectoriesArrayDictionary,trackedArrays,leftRight='right')
        LimitLeftOrRightOfSBF(priorToAscentArrayDictionary,trackedArrays,leftRight='right')
        RemoveAllNanParcels(trajectoriesArrayDictionary,priorToAscentArrayDictionary)

    #subsetting time range
    if subsetting_time is not None:
        SubsetTime(trajectoriesArrayDictionary,priorToAscentArrayDictionary,subsetting_time)
        RemoveAllNanParcels(trajectoriesArrayDictionary,priorToAscentArrayDictionary)

    #subsetting supersets
    for arrayDictionary in [trajectoriesArrayDictionary,priorToAscentArrayDictionary]:
        arrayDictionary=SubsetDataParcelType(arrayDictionary=arrayDictionary,
                                                         parcelType_original='CL', parcelType_subset='SBF')
        arrayDictionary=SubsetDataParcelType(arrayDictionary=arrayDictionary,
                                                         parcelType_original='CL', parcelType_subset='ColdPool')
        
    return trajectoriesArrayDictionary, priorToAscentArrayDictionary

def LimitLeftOrRightOfSBF(dictionary,trackedArrays,
                          leftRight='right'):
    leftRightIdentifier=1 if leftRight=='right' else -1
    
    for parcelType in dictionary:
        for parcelDepth in dictionary[parcelType]:
            leftRightList = trackedArrays[parcelType][parcelDepth][:,4] 
            mask = leftRightList == leftRightIdentifier
            for varName in dictionary[parcelType][parcelDepth]:
                dictionary[parcelType][parcelDepth][varName]=dictionary[parcelType][parcelDepth][varName][:,mask]
    # return dictionary
    
def SubsetTime(trajectoriesArrayDictionary,priorToAscentArrayDictionary,
               subsetting_time):
    times=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
    mask = (times >= (subsetting_time[0] or -np.inf)) & (times <= (subsetting_time[1] or np.inf))
    timeIndices=np.arange(ModelData.Ntime)[mask]
    
    for parcelType in trajectoriesArrayDictionary:
        for parcelDepth in trajectoriesArrayDictionary[parcelType]:
            for varName in trajectoriesArrayDictionary[parcelType][parcelDepth]:
                data_1 = trajectoriesArrayDictionary[parcelType][parcelDepth][varName]
                data_2 = priorToAscentArrayDictionary[parcelType][parcelDepth][varName]
                
                first_valid = np.argmax(~np.isnan(data_1), axis=0)
                keep_mask = np.isin(first_valid, timeIndices)
                data_1[:, ~keep_mask] = np.nan
                data_2[:, ~keep_mask] = np.nan
                
    # return trajectoriesArrayDictionary,priorToAscentArrayDictionary

def RemoveAllNanParcels(trajectoriesArrayDictionary,priorToAscentArrayDictionary):
    """
    use after applying SubsetTime()
    """
    for parcelType in trajectoriesArrayDictionary:
        for parcelDepth in trajectoriesArrayDictionary[parcelType]:
            for varName in trajectoriesArrayDictionary[parcelType][parcelDepth]:
                data_1 = trajectoriesArrayDictionary[parcelType][parcelDepth][varName]
                data_2 = priorToAscentArrayDictionary[parcelType][parcelDepth][varName]
                
                keep_mask = np.any(~np.isnan(data_1), axis=0)
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName] = data_1[:, keep_mask]
                priorToAscentArrayDictionary[parcelType][parcelDepth][varName] = data_2[:, keep_mask]
                
    return trajectoriesArrayDictionary, priorToAscentArrayDictionary

def SubtractMeanData(trajectoriesArrayDictionary,priorToAscentArrayDictionary,dataName_Global):

    if dataName_Global != 'Variables': return
        
    DataManager_CalculateMeans = DataManager_Class(mainDirectory, scratchDirectory, ModelData, 
                                                   dataType="CalculateMoreVariables",dataName="CalculateMeans",dtype='float32',
                                                   verbose=False)
    
    [inputDataDirectory,dataName] = CallVariable(ModelData, DataManager_CalculateMeans, 'all_times', 'qv_mean',loadData=False) 
    meanData = DataManager.GetTimestepData_V2(inputDataDirectory, 'all_times', dataName=dataName)
    
    varNames = GetMeanVarNames()
    for varName in varNames:
        varNameMean = varName.lower()+'_mean'
        if varNameMean not in meanData: continue        
        varMean = meanData[varNameMean][:]  # shape (t, Nz)
        for dictionary in [trajectoriesArrayDictionary, priorToAscentArrayDictionary]:
            for parcelType in dictionary:
                for parcelDepth in dictionary[parcelType]:
                    d = dictionary[parcelType][parcelDepth]
                    zIndexes = d['Z']
                    validMask = ~np.isnan(zIndexes)
    
                    perturbation = np.full(d[varName].shape, np.nan)
                    tIndexes = np.arange(varMean.shape[0])[:, np.newaxis] * np.ones_like(zIndexes, dtype=int)
                    
                    perturbation[validMask] = d[varName][validMask] - varMean[tIndexes[validMask], zIndexes[validMask].astype(int)]    
                    dictionary[parcelType][parcelDepth][varName+'_perturbation'] = perturbation
                    
    #close data
    meanData.close()

def SubtractSBF_XLocation(trajectoriesArrayDictionary, priorToAscentArrayDictionary, SBF_XLocations):

    SBF_XLocations = np.asarray(SBF_XLocations, dtype=float)

    for dictionary in [trajectoriesArrayDictionary, priorToAscentArrayDictionary]:
        for parcelType in dictionary:
            for parcelDepth in dictionary[parcelType]:

                d = dictionary[parcelType][parcelDepth]

                xData = d['X']
                validMask = ~np.isnan(xData)

                tIndexes = np.arange(xData.shape[0])[:, np.newaxis] * np.ones_like(xData, dtype=int)

                sbfForParcels = SBF_XLocations[tIndexes]

                validMask = validMask & ~np.isnan(sbfForParcels)

                xPerturbation = np.full(xData.shape, np.nan)
                xPerturbation[validMask] = xData[validMask] - sbfForParcels[validMask]

                d['X_SBF_Relative'] = xPerturbation

def FixDetrainmentSign(dictionary):
    for parcelType in dictionary:
        for parcelDepth in dictionary[parcelType]:
            for varName in dictionary[parcelType][parcelDepth]:
                PROCESSED_string = 'PROCESSED_' if 'PROCESSED_' in varName else ""
                DivideMassFlux_string = '_DivideMassFlux' if '_DivideMassFlux' in varName else ""
                if f"{PROCESSED_string}D{DivideMassFlux_string}_" in varName:
                    dictionary[parcelType][parcelDepth][varName] *= -1

def CalculateNetEntrainment(dictionary):
    for parcelType in dictionary:
        for parcelDepth in dictionary[parcelType]:
            variables = dictionary[parcelType][parcelDepth]
            varNames = list(variables.keys())
            entrainment_varNames = [varName for varName in varNames if "_E_" in varName]
            for entrainment_varName in entrainment_varNames:
                detrainment_varName = entrainment_varName.replace('_E_', '_D_')
                net_varName = entrainment_varName.replace('_E_', '_NET_')
                variables[net_varName] = variables[entrainment_varName] - variables[detrainment_varName]

def SubsetDataParcelType(arrayDictionary,parcelType_original='CL', parcelType_subset='SBF'):
    """
    In arrayDictionary, the parcelType CL is a superset of SBF and ColdPool.
    This function allows simple "isin" mask to create new parcelTypes for the subset of each superset,
    using indicies marking each subset within trackedArrays (loaded below), as calculated from the SubsetParcels code.
    """

    trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class,verbose=False)
    
    original = trackedArrays[parcelType_original]['ALL']
    subset = trackedArrays[parcelType_subset]['ALL']
    
    mask = np.isin(original[:,0], subset[:,0])
    
    arrayDictionary[parcelType_subset] = {}
    
    for inner_key1 in arrayDictionary[parcelType_original]:
        arrayDictionary[parcelType_subset][inner_key1] = {}
        for inner_key2 in arrayDictionary[parcelType_original][inner_key1]:
            varData = arrayDictionary[parcelType_original][inner_key1][inner_key2]
            arrayDictionary[parcelType_subset][inner_key1][inner_key2] = varData[:, mask]
    
    return arrayDictionary

In [ ]:
#Loading Data
if not running:
    subsetting_RightOfSBF=False; #subsetting_RightOfSBF=True
    subsetting_time=None; #subsetting_time=(12,None)
    
    [trajectoriesArrayDictionary_Variables,priorToAscentArrayDictionary_Variables]\
    =LoadAndProcessData(subsetting_RightOfSBF=subsetting_RightOfSBF,subsetting_time=subsetting_time,dataName='Variables')
    [trajectoriesArrayDictionary_UpdraftArea,priorToAscentArrayDictionary_Variables]\
    =LoadAndProcessData(subsetting_RightOfSBF=subsetting_RightOfSBF,subsetting_time=subsetting_time,dataName='UpdraftArea')
    [trajectoriesArrayDictionary_Entrainment,priorToAscentArrayDictionary_Entrainment]\
    =LoadAndProcessData(subsetting_RightOfSBF=subsetting_RightOfSBF,subsetting_time=subsetting_time,dataName='Entrainment')

In [ ]:
#PLOTTING FUNCTIONS
def plotRegression(ax, x_transformed, y_transformed, n_bins=30, color='black'):
    from scipy import stats

    # Regression directly in transformed space (both x and y already log-transformed)
    slope, intercept, r, p, se = stats.linregress(x_transformed, y_transformed)
    x_line = np.linspace(x_transformed.min(), x_transformed.max(), 200)
    y_line = slope * x_line + intercept  # straight line in transformed space

    # Binned median in transformed space
    bin_edges = np.linspace(x_transformed.min(), x_transformed.max(), n_bins + 1)
    bin_centers, medians, q25, q75 = [], [], [], []
    for i in range(n_bins):
        in_bin = (x_transformed >= bin_edges[i]) & (x_transformed < bin_edges[i+1])
        if in_bin.sum() > 10:
            bin_centers.append((bin_edges[i] + bin_edges[i+1]) / 2)
            medians.append(np.median(y_transformed[in_bin]))
            q25.append(np.percentile(y_transformed[in_bin], 25))
            q75.append(np.percentile(y_transformed[in_bin], 75))

    bin_centers = np.array(bin_centers)

    ax.plot(x_line, y_line, color='white', linewidth=3.5, linestyle='--', zorder=3)
    ax.plot(x_line, y_line, color=color,   linewidth=2,   linestyle='--', zorder=4,
            label=f'OLS r={r:.2f}, p={p:.1e}')
    ax.legend(fontsize=9, loc='upper right', framealpha=0.6)
    return {'slope': slope, 'intercept': intercept, 'r': r, 'p': p}

def signed_log(arr, linthresh=1e-6):
    if (arr < 0).any():
        return np.sign(arr) * np.log10(1 + np.abs(arr) / linthresh)
    return np.log10(arr)

def set_x_ticks(ax, x_plot, x_transformed):
    if (x_plot < 0).any():
        ticks = np.array([-1e-2, -1e-3, -1e-4, 0, 1e-4, 1e-3, 1e-2])
        ax.set_xticks(signed_log(ticks))
        ax.set_xticklabels([f'{v:.0e}' for v in ticks], rotation=30)
    else:
        powers = np.arange(np.floor(x_transformed.min()), np.ceil(x_transformed.max()) + 1)
        ax.set_xticks(powers)
        ax.set_xticklabels([f'$10^{{{int(p)}}}$' for p in powers], rotation=30)

In [ ]:
#PLOTTING

# ── MAIN ──────────────────────────────────────────────────────────────────────
parcelType = 'CL'
# varName_variable='UPDRAFT_AREA_g'
# varName_variable='UPDRAFT_AREA_c'
# varName_variable = 'QV_perturbation'
# varName_variable='THETA_v_perturbation'

data_variables = (trajectoriesArrayDictionary_Variables[parcelType]['ALL']
                  if varName_variable in trajectoriesArrayDictionary_Variables[parcelType]['ALL']
                  else trajectoriesArrayDictionary_UpdraftArea[parcelType]['ALL'])[varName_variable]
updraftTypeCondition = varName_variable not in trajectoriesArrayDictionary_Variables[parcelType]['ALL']

for entrainmentType in ['E', 'D', 'NET']:
    for updraftType in ['g', 'c']:
        if updraftTypeCondition and f'_{updraftType}' not in varName_variable: continue

        varName_entrainment = f'PROCESSED_{entrainmentType}_DivideMassFlux_{updraftType}'
        data_entrainment = trajectoriesArrayDictionary_Entrainment['CL']['ALL'][varName_entrainment]

        # ── Data prep ─────────────────────────────────────────────────────────
        x = data_variables.flatten();  y = data_entrainment.flatten()
        mask = np.isfinite(x) & np.isfinite(y)
        y_mask = mask & (np.isfinite(y) if entrainmentType == 'NET' else (y > 1e-5))
        x_plot, y_plot = x[y_mask], y[y_mask]
        x_transformed = signed_log(x_plot)

        # ── Plot ──────────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(8, 6))
        has_neg_y = (y_plot <= 0).any()
        y_transformed = signed_log(y_plot) if has_neg_y else np.log10(y_plot)

        h = ax.hist2d(x_transformed, y_transformed, bins=100, cmap='turbo', density=True, norm=LogNorm())
        plt.colorbar(h[3], ax=ax, label='Probability Density')

        if has_neg_y:
            y_ticks = np.array([-1e-2, -1e-3, -1e-4, 0, 1e-4, 1e-3, 1e-2])
            ax.set_yticks(signed_log(y_ticks))
            ax.set_yticklabels([f'{v:.0e}' for v in y_ticks])
        else:
            ax.set_yscale('log')

        plotRegression(ax, x_transformed, y_transformed, n_bins=30, color='black')
        set_x_ticks(ax, x_plot, x_transformed)
        ax.set(xlabel=varName_variable, ylabel=varName_entrainment,
               title=f'Joint Distribution: {varName_variable} vs {varName_entrainment}')
        plt.tight_layout()
        plt.show()